# Sentiment Timeline Analysis

This notebook presents the sentiment timeline analysis of perceived ratings over times for courses (difficulty + quality) lining up with major LLM release dates (GPT)


In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import sys

In [ ]:
cwd = Path.cwd().resolve()
root_dir = next((path for path in [cwd, *cwd.parents] if (path / "src").exists() and (path / "data" / "rmp_ucsd_reviews.csv").exists()), None)
if root_dir is None:
    raise FileNotFoundError("Could not find the project root from the current notebook working directory.")
data_dir = root_dir / "data" / "rmp_ucsd_reviews.csv"
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))
df = pd.read_csv(data_dir)



In [ ]:
COLUMNS_TO_KEEP = [
    "date",
    "class",
    "qualityRating",
    "difficultyRatingRounded",
    "grade",
    "comment",
]
df = df[COLUMNS_TO_KEEP].copy()
valid_course_pattern = r"^[A-Z]{3,}[0-9]+[A-Z]?$"
df = df[df["class"].str.match(valid_course_pattern, na=False)].copy()
df

In [ ]:
df = df.dropna().copy()
df = df.rename(columns={
    "qualityRating": "quality",
    "clarityRatingRounded": "clarity",
    "difficultyRatingRounded": "difficulty",
})
df

## Department Averages



In [ ]:
from src.sentiment import add_department_codes, normalize_dates, summarize_department_ratings

df_clean = normalize_dates(df=df)
df_clean = add_department_codes(df_clean)

department_summary = summarize_department_ratings(df_clean)
department_summary.round(3)


## Major Department Plots


In [ ]:
from IPython.display import display

from src.sentiment import load_releases, overlay_releases

releases_df = load_releases(root_dir / "data" / "chatgpt_model_updates.csv")
analysis_start = pd.Timestamp("2016-01-01")
analysis_end = pd.Timestamp("2026-12-31")

from src.sentiment import (
    build_department_ratings,
    filter_departments,
    major_departments,
    plot_department_ratings,
    plot_release_period_ratings,
    smooth_department_ratings,
    summarize_release_period_ratings,
)

departments = major_departments(
    df_clean,
    include=["ECE", "HUM", "CSE", "MATH"],
    top_n=8,
)
print(departments)

df_department_reviews = filter_departments(
    df=df_clean,
    departments=departments,
    start_date=analysis_start,
    end_date=analysis_end,
)
df_department = build_department_ratings(df_department_reviews)
df_department = smooth_department_ratings(df_department)

for department in departments:
    fig, ax = plot_department_ratings(df_department, department)
    overlay_releases(ax, releases_df)
    plt.show()

for department in departments:
    department_reviews = df_department_reviews[df_department_reviews["department"] == department].copy()
    department_release_summary = summarize_release_period_ratings(
        department_reviews,
        releases_df=releases_df,
    )
    display(
        department_release_summary[[
            "release_period",
            "quality_mean",
            "difficulty_mean",
            "review_count",
        ]].round(3)
    )

    fig, ax = plot_release_period_ratings(
        department_release_summary,
        releases_df=releases_df,
        title=f"{department}: Average Ratings by LLM Release Period",
        start_date=analysis_start,
        end_date=analysis_end,
    )
    overlay_releases(ax, releases_df)
    plt.show()

selected_departments_release_summary = summarize_release_period_ratings(
    df_department_reviews,
    releases_df=releases_df,
)
display(
    selected_departments_release_summary[[
        "release_period",
        "quality_mean",
        "difficulty_mean",
        "review_count",
    ]].round(3)
)

fig, ax = plot_release_period_ratings(
    selected_departments_release_summary,
    releases_df=releases_df,
    title="Selected UCSD Departments Combined: Average Ratings by LLM Release Period",
    start_date=analysis_start,
    end_date=analysis_end,
)
overlay_releases(ax, releases_df)
plt.show()



In [ ]:
from src.sentiment import (
    count_department_release_behaviors,
    summarize_department_release_behaviors,
)

classification_start = pd.Timestamp("2024-01-01")
classification_end = pd.Timestamp("2025-12-31")

department_behavior_summary = summarize_department_release_behaviors(
    df=df_clean,
    releases_df=releases_df,
    start_date=classification_start,
    end_date=classification_end,
    stable_threshold=0.15,
)
display(department_behavior_summary.round(3))

department_behavior_counts = count_department_release_behaviors(
    department_behavior_summary
)
display(department_behavior_counts)


In [ ]:
from src.sentiment import (
    build_overall_ratings,
    plot_overall_ratings,
    smooth_overall_ratings,
    summarize_release_period_ratings,
    plot_release_period_ratings,
)

df_ucsd_reviews = df_clean.loc[
    (df_clean["date"] >= analysis_start)
    & (df_clean["date"] <= analysis_end)
].copy()

df_ucsd_overall = build_overall_ratings(df_ucsd_reviews)
df_ucsd_overall = smooth_overall_ratings(df_ucsd_overall)

fig, ax = plot_overall_ratings(
    df_ucsd_overall,
    title="UCSD Overall RMP Ratings Over Time (Smoothed)",
)
overlay_releases(ax, releases_df)
plt.show()

ucsd_release_summary = summarize_release_period_ratings(
    df_ucsd_reviews,
    releases_df=releases_df,
)
display(
    ucsd_release_summary[[
        "release_period",
        "quality_mean",
        "difficulty_mean",
        "review_count",
    ]].round(3)
)

fig, ax = plot_release_period_ratings(
    ucsd_release_summary,
    releases_df=releases_df,
    title="UCSD Overall: Average Ratings by LLM Release Period",
    start_date=analysis_start,
    end_date=analysis_end,
)
overlay_releases(ax, releases_df)
plt.show()



In [ ]:
from src.sentiment import normalize_dates, filter_courses, plot_course_ratings

course = "PHYS2A"

df_clean = normalize_dates(df=df)
df_filtered = filter_courses(df=df_clean,
                             course=course,
                             start_date=pd.Timestamp("2016-01-01"),
                             end_date=pd.Timestamp("2026-12-31"))
fig, ax = plot_course_ratings(df=df_filtered, course=course)
plt.show()

In [ ]:
from src.sentiment import smooth_ratings, plot_smoothed_ratings

df_smoothed = smooth_ratings(df_filtered)

fig, ax = plot_smoothed_ratings(df_smoothed, course)
plt.show()

In [ ]:
from src.sentiment import load_releases, overlay_releases

releases_df = load_releases(root_dir / "data" / "chatgpt_model_updates.csv")

fig, ax = plot_smoothed_ratings(df_smoothed, course)
overlay_releases(ax, releases_df)

plt.show()

In [ ]:
course_list = ["MATH20C", "PHYS2A", "CHEM6A", "ECE35", "ECE45", "HUM3", "MMW11"]

for course in course_list:
    df_filtered = filter_courses(df=df_clean,
                             course=course,
                             start_date=pd.Timestamp("2016-01-01"),
                             end_date=pd.Timestamp("2026-12-31"))
    df_smoothed = smooth_ratings(df_filtered)
    fig, ax = plot_smoothed_ratings(df_smoothed, course)
    overlay_releases(ax, releases_df)

plt.show()
